# 04 — Chia train/dev/test

Các crop cùng họ tên tệp gốc luôn nằm trong cùng một tập. Nhóm được tạo từ basename toàn cục sau khi bỏ `.rf.<32 ký tự hex>`; tên tệp thực tế không có hậu tố augmentation khác. Chia nhóm theo nguồn và mệnh giá bằng 10 folds cố định: fold 0 là test, fold 1 là dev, folds 2–9 là train. Notebook chỉ ghi bảng phân chia; không thay đổi ảnh.

In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

cwd = Path.cwd().resolve()
root = cwd if (cwd / "DATA_RAW").is_dir() else cwd.parents[1]
metadata_dir = root / "DATA_CROP" / "metadata"
images = pd.read_csv(metadata_dir / "unique_crops.csv", dtype=str).fillna("")
sources = ["banknotes_v2", "money_v1", "blindtrix_v1", "currency_v6"]
denominations = ["10000", "20000", "50000", "100000", "200000", "500000"]
split_order = ["train", "dev", "test"]


In [2]:
group_ids = images["source_rel_path"].map(
    lambda path: re.sub(r"\.rf\.[0-9a-f]{32}$", "", Path(path).stem.casefold())
)
strata = images["dataset_id"].astype(str) + "|" + images["denomination"].astype(str)
splitter = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=2312026)
fold_ids = np.full(len(images), -1, dtype=int)
for fold_id, (_, validation_rows) in enumerate(
    splitter.split(np.zeros(len(images)), strata, groups=group_ids)
):
    fold_ids[validation_rows] = fold_id
split_names = np.where(fold_ids == 0, "test", np.where(fold_ids == 1, "dev", "train"))
split_crops = images.copy()
split_crops["group_id"] = group_ids
split_crops["split"] = split_names


In [3]:
split_crops.to_csv(metadata_dir / "split_crops.csv", index=False, encoding="utf-8-sig")

split_counts = (
    split_crops.groupby(["dataset_id", "denomination", "split"])
    .size()
    .reindex(
        pd.MultiIndex.from_product(
            [sources, denominations, split_order],
            names=["dataset_id", "denomination", "split"],
        ),
        fill_value=0,
    )
    .rename("count")
    .reset_index()
)
split_counts.to_csv(metadata_dir / "split_counts.csv", index=False, encoding="utf-8-sig")

overall_counts = split_crops["split"].value_counts().reindex(split_order, fill_value=0)
overall_table = pd.DataFrame({"count": overall_counts, "percent": (overall_counts / len(split_crops) * 100).round(2)})
denomination_table = pd.crosstab(split_crops["denomination"], split_crops["split"]).reindex(index=denominations, columns=split_order, fill_value=0)
source_table = pd.crosstab(split_crops["dataset_id"], split_crops["split"]).reindex(index=sources, columns=split_order, fill_value=0)

display(overall_table)
display(denomination_table)
display(source_table)


,count,percent
split,,
train,12494,80.03
dev,1564,10.02
test,1553,9.95


split,train,dev,test
denomination,,,
10000,2299,315,281
20000,1916,238,224
50000,1646,195,223
100000,1918,250,247
200000,2525,309,317
500000,2190,257,261


split,train,dev,test
dataset_id,,,
banknotes_v2,1683,208,206
money_v1,5832,726,730
blindtrix_v1,3945,498,485
currency_v6,1034,132,132
